# Stage 1 data check
Run all. The last cell prints a short summary: copy all of its output back to Claude.

In [ ]:
MOUNT_DRIVE = True
SNAP_BEFORE_CLOSE_MIN = 25   # snapshot = last hourly candle ending at least this long before close (~8:00 ET)
MAX_SPREAD = 0.03            # quote counts as usable if ask - bid <= 3 cents
MIN_STRIKES, MIN_VOLUME = 7, 50_000

import pandas as pd, numpy as np
from pathlib import Path
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW = Path("/content/drive/MyDrive/macro-news-pricing/data/raw")
else:
    RAW = Path("/content/data/raw")
ev = pd.read_parquet(RAW / "kalshi_events.parquet")
mk = pd.read_parquet(RAW / "kalshi_markets.parquet")
cd = pd.read_parquet(RAW / "kalshi_candles.parquet")
nc = pd.read_parquet(RAW / "nowcast_cpi.parquet")
act = pd.read_parquet(RAW / "nowcast_actuals.parquet")
fred = pd.read_parquet(RAW / "fred_daily.parquet")

In [ ]:
MON = {m: i for i, m in enumerate("JAN FEB MAR APR MAY JUN JUL AUG SEP OCT NOV DEC".split(), 1)}
def target_month(event_ticker):
    code = event_ticker.split("-")[1][:5]          # e.g. 26AUG (drops suffixes like 25DECT)
    return f"20{code[:2]}-{MON[code[2:5]]:02d}"

ev = ev[~ev.no_report].copy()
ev["target_month"] = ev.event_ticker.map(target_month)
ev["release_date"] = pd.to_datetime(ev.release_date_et)
ev["series_kind"] = np.where(ev.event_ticker.str.contains("CORE"), "core_cpi", "cpi")

# 1. Kalshi snapshot quality at ~8:00 ET on release day
cd = cd.merge(mk[["ticker", "event_ticker", "close_time"]], on="ticker")
snap = cd[cd.end_time <= cd.close_time - pd.Timedelta(minutes=SNAP_BEFORE_CLOSE_MIN)]
snap = snap.sort_values("end_time").groupby("ticker").tail(1)
snap["spread"] = snap.yes_ask_close - snap.yes_bid_close
snap["hours_before"] = (snap.close_time - snap.end_time).dt.total_seconds() / 3600
q = snap.groupby("event_ticker").agg(snap_strikes=("ticker", "size"),
        usable_share=("spread", lambda s: (s <= MAX_SPREAD).mean()),
        med_spread=("spread", "median"), max_hours_before=("hours_before", "max"))
ev = ev.merge(q, left_on="event_ticker", right_index=True, how="left")

# 2. Nowcast as of the business day before release
def nowcast_asof(r):
    d = nc[(nc.target_month == r.target_month) & (nc.date < r.release_date)].dropna(subset=[r.series_kind])
    if d.empty:
        return pd.Series({"nowcast": np.nan, "nowcast_age_days": np.nan})
    last = d.iloc[-1]
    return pd.Series({"nowcast": last[r.series_kind], "nowcast_age_days": (r.release_date - last.date).days})
ev = ev.join(ev.apply(nowcast_asof, axis=1))

# 3. Realized print cross-check: Kalshi ladder vs Cleveland "actual", rounded to 0.1
a = act.pivot_table(index="target_month", columns="measure", values="value")
ev["cleveland_actual"] = [a.at[t, k] if t in a.index and k in a.columns else np.nan
                          for t, k in zip(ev.target_month, ev.series_kind)]
ev["print_match"] = (ev.cleveland_actual.round(1) - ev["print"]).abs() < 1e-9

# 4. 2-year yield available on release day and the trading day before
f = fred.set_index("date").DGS2.dropna()
ev["dgs2_ok"] = [d in f.index and (f.index < d).any() for d in ev.release_date]
ev["d2y_bp"] = [100 * (f[d] - f[f.index < d].iloc[-1]) if ok else np.nan
                for d, ok in zip(ev.release_date, ev.dgs2_ok)]

ev["liquid"] = (ev.n_strikes >= MIN_STRIKES) & (ev.volume >= MIN_VOLUME) & (ev.usable_share >= 0.6)
ev.to_parquet(RAW / "qa_events.parquet", index=False)

In [ ]:
pd.set_option("display.width", 250, "display.max_rows", 300, "display.max_columns", 30)
for kind, g in ev.groupby("series_kind"):
    print(f"\n=== {kind}: {len(g)} releases ({g.release_date.min():%Y-%m} to {g.release_date.max():%Y-%m})")
    print("liquid:", int(g.liquid.sum()), "| print exact:", int((g.print_flag == "exact").sum()),
          "| ladder inconsistent:", int((~g.consistent).sum()),
          "| no snapshot:", int(g.snap_strikes.isna().sum()),
          "| nowcast missing:", int(g.nowcast.isna().sum()),
          "| print vs Cleveland mismatch:", int((~g.print_match & g.cleveland_actual.notna() & g["print"].notna()).sum()),
          "| 2y missing:", int((~g.dgs2_ok).sum()))
    print(g[["event_ticker", "release_date_et", "n_strikes", "volume", "usable_share", "med_spread",
             "print", "print_flag", "cleveland_actual", "nowcast", "nowcast_age_days", "d2y_bp", "liquid"]]
          .round(3).to_string(index=False))

## Fixes and snapshot comparison
1. Fill prints the ladder could not pin down (print outside the strikes) with the Cleveland actual rounded to 0.1, only where it agrees with the ladder bound.
2. Drop Nov 2025 (post-shutdown release; Kalshi and Cleveland disagree on what the monthly change was).
3. Compare quote quality at several pre-release snapshot times, measured only on strikes that are still in play (mid between 5 and 95 cents). The snapshot is chosen on quote quality alone, never on forecast results.

In [ ]:
DROP = {"KXCPI-25NOV", "KXCPICORE-25NOV"}
START = "2022-03-01"
SNAPS = {"T-1 16:00": (-1, 16), "T-1 20:00": (-1, 20), "T 00:00": (0, 0), "T 07:00": (0, 7), "T 08:00": (0, 8)}

e = ev[(ev.release_date >= START) & ~ev.event_ticker.isin(DROP)].copy()
cl = e.cleveland_actual.round(1)
fill_ok = ((e.print_flag == "below_ladder") & (cl <= e.min_strike)) | \
          ((e.print_flag == "above_ladder") & (cl > e.max_strike)) | \
          ((e.print_flag == "interval") & (cl >= e.print_lo) & (cl <= e.print_hi))
e["print_final"] = e["print"].where(e.print_flag == "exact", cl.where(fill_ok))
e["print_source"] = np.where(e.print_flag == "exact", "ladder", np.where(fill_ok, "cleveland", "MISSING"))

cdm = cd.merge(mk[["ticker", "event_ticker"]].rename(columns={"event_ticker": "ev_t"}), on="ticker") \
    if "event_ticker" not in cd.columns else cd.rename(columns={"event_ticker": "ev_t"})
cdm["mid"] = (cdm.yes_bid_close + cdm.yes_ask_close) / 2
cdm["spread"] = cdm.yes_ask_close - cdm.yes_bid_close
cdm["end_et"] = cdm.end_time.dt.tz_convert("America/New_York")

rows = []
for r in e.itertuples():
    g = cdm[cdm.ev_t == r.event_ticker]
    rel = pd.Timestamp(r.release_date).tz_localize("America/New_York")
    for name, (dd, hh) in SNAPS.items():
        t = rel + pd.Timedelta(days=dd, hours=hh)
        if dd == -1:  # previous business day
            t = (rel - pd.offsets.BDay(1)) + pd.Timedelta(hours=hh)
        s = g[g.end_et <= t].sort_values("end_et").groupby("ticker").tail(1)
        s = s[(rel.normalize() + pd.Timedelta(hours=8, minutes=25)) - s.end_et <= pd.Timedelta(days=10)]
        live = s[(s.mid >= 0.05) & (s.mid <= 0.95)]
        rows.append({"event_ticker": r.event_ticker, "snap": name, "live_strikes": len(live),
                     "usable_share": (live.spread <= MAX_SPREAD).mean() if len(live) else np.nan,
                     "med_spread": live.spread.median(), "quote_missing": int(s.spread.isna().sum())})
sq = pd.DataFrame(rows)
summary = sq.groupby("snap").agg(releases=("event_ticker", "nunique"),
            med_live_strikes=("live_strikes", "median"), mean_usable=("usable_share", "mean"),
            med_spread=("med_spread", "median"),
            ok_releases=("usable_share", lambda s: int((s >= 0.6).sum()))).reindex(SNAPS)
e.to_parquet(RAW / "qa_events_final.parquet", index=False)
sq.to_parquet(RAW / "qa_snapshots.parquet", index=False)

print(f"Sample from {START}: {len(e)} releases | print from ladder {int((e.print_source=='ladder').sum())}, "
      f"filled from Cleveland {int((e.print_source=='cleveland').sum())}, missing {int((e.print_source=='MISSING').sum())}")
print(e.loc[e.print_source != "ladder", ["event_ticker", "print_flag", "min_strike", "max_strike",
      "cleveland_actual", "print_final", "print_source"]].to_string(index=False))
print("\nQuote quality on in-play strikes, by snapshot time (ET):")
print(summary.round(3).to_string())
by_year = sq.assign(yr=sq.event_ticker.map(lambda x: "20" + x.split("-")[1][:2])) \
            .pivot_table(index="yr", columns="snap", values="usable_share", aggfunc="mean").reindex(columns=list(SNAPS))
print("\nMean usable share by year:")
print(by_year.round(2).to_string())
